# run_future_dpwm notebook

This notebook embeds the full code from `run_future_dpwm.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Build future rainplusmelt + PET from bias-corrected EURO-CORDEX outputs, then run DPWM
using the same calibrated parameters as the baseline.

Inputs (per basin, under --bc_dir, from bias_correction.py):
  pr_bc_daily_{basin}.csv   (columns: date, pr_raw, pr_bc)
  tas_bc_daily_{basin}.csv  (columns: date, tas_raw, tas_bc)
  pet_future_bc_{basin}.csv (columns: date, pet_mm_day)

Rainplusmelt uses the same snow routine as prepare_eastream_inputs.py (frac_snow from
estreams_hydrometeo_signatures.csv, DDF=5.9).

Outputs:
  Results/forcing_future/rainplusmelt.csv
  Results/forcing_future/PET.csv
  Results/discharge_components_future/Q_total_all_basins.csv
  Results/discharge_components_future/discharge_components_{basin}.csv (per basin)

Run (PowerShell, from project root):
  python run_future_dpwm.py
  python run_future_dpwm.py --basins CH000127,DESN1585,FR002785,GB000215,SE000197,ES000700
  python run_future_dpwm.py --only_forcing
  python run_future_dpwm.py --only_simulate
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from dpwm_model import DPWM
from prepare_eastream_inputs import (
    SIGNATURES_PATH,
    compute_rainplusmelt_components_for_basin,
    get_frac_snow_per_basin,
)
from calibration_io import BASIN_IDS, WARM_SPAN_DAYS, load_preferred_calibration_df


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    p.add_argument("--root", type=Path, default=Path("."), help="Project root (default: current directory).")
    p.add_argument(
        "--bc_dir",
        type=Path,
        default=None,
        help="Bias-corrected outputs (default: <root>/Results/eurocordex_bias_corrected).",
    )
    p.add_argument(
        "--forcing_out",
        type=Path,
        default=None,
        help="Where to write future rainplusmelt.csv and PET.csv (default: <root>/Results/forcing_future).",
    )
    p.add_argument(
        "--discharge_out",
        type=Path,
        default=None,
        help="Where to write future discharge CSVs (default: <root>/Results/discharge_components_future).",
    )
    p.add_argument(
        "--signatures_csv",
        type=Path,
        default=None,
        help="eStreams signatures (default: <root>/data/csv_estream/estreams_hydrometeo_signatures.csv).",
    )
    p.add_argument(
        "--basins",
        default="",
        help=(
            "Comma-separated basin IDs. Default: auto-detect from bias-corrected files in --bc_dir "
            f"(expected names like pr_bc_daily_{{basin}}.csv). Target list: {','.join(BASIN_IDS)}"
        ),
    )
    p.add_argument("--warm_span_days", type=int, default=WARM_SPAN_DAYS, help="DPWM warm-up length (default: 365).")
    p.add_argument("--only_forcing", action="store_true", help="Only build rainplusmelt + PET CSVs.")
    p.add_argument("--only_simulate", action="store_true", help="Only run DPWM (expects forcing CSVs to exist).")
    return p.parse_known_args()[0]


def basin_has_bc_forcing(bc_dir: Path, basin_id: str) -> bool:
    return all(
        (bc_dir / name).exists()
        for name in (
            f"pr_bc_daily_{basin_id}.csv",
            f"tas_bc_daily_{basin_id}.csv",
            f"pet_future_bc_{basin_id}.csv",
        )
    )


def discover_basins_from_bc_dir(bc_dir: Path, preferred: list[str] | None = None) -> list[str]:
    """Basins with all three bias-corrected forcing files, in preferred order when given."""
    if not bc_dir.is_dir():
        return []
    found: set[str] = set()
    for p in bc_dir.glob("pr_bc_daily_*.csv"):
        basin_id = p.name[len("pr_bc_daily_") : -4]
        if basin_has_bc_forcing(bc_dir, basin_id):
            found.add(basin_id)
    order = preferred if preferred is not None else sorted(found)
    return [b for b in order if b in found]


def resolve_basins(
    requested: list[str],
    bc_dir: Path,
    forcing_out: Path,
    only_simulate: bool,
) -> list[str]:
    if only_simulate:
        rain_path = forcing_out / "rainplusmelt.csv"
        if not rain_path.exists():
            raise FileNotFoundError(f"Missing {rain_path}; run without --only_simulate first.")
        rain_df = pd.read_csv(rain_path, nrows=1)
        available = [c for c in rain_df.columns if c != "date"]
        if not requested:
            return available
        missing = [b for b in requested if b not in available]
        for b in missing:
            print(f"Skipping {b}: not in {rain_path}")
        out = [b for b in requested if b in available]
        if not out:
            raise ValueError("No requested basins found in existing forcing CSV.")
        return out

    available = discover_basins_from_bc_dir(bc_dir, preferred=BASIN_IDS)
    if not requested:
        if not available:
            raise FileNotFoundError(
                f"No complete bias-corrected forcing found in {bc_dir}. "
                "Run bias_correction.py per basin first (pr, tas, pet_future_bc)."
            )
        print(f"Auto-detected basins with bias-corrected forcing: {', '.join(available)}")
        return available

    out: list[str] = []
    for b in requested:
        if basin_has_bc_forcing(bc_dir, b):
            out.append(b)
        else:
            print(f"Skipping {b}: missing bias-corrected files in {bc_dir}")
    if not out:
        raise FileNotFoundError(
            f"None of the requested basins have complete bias-corrected forcing in {bc_dir}."
        )
    return out


def _read_bc_forcing(bc_dir: Path, basin_id: str) -> pd.DataFrame:
    pr_path = bc_dir / f"pr_bc_daily_{basin_id}.csv"
    tas_path = bc_dir / f"tas_bc_daily_{basin_id}.csv"
    pet_path = bc_dir / f"pet_future_bc_{basin_id}.csv"
    for path in (pr_path, tas_path, pet_path):
        if not path.exists():
            raise FileNotFoundError(f"Missing required file for basin {basin_id}: {path}")

    pr = pd.read_csv(pr_path)
    tas = pd.read_csv(tas_path)
    pet = pd.read_csv(pet_path)
    pr["date"] = pd.to_datetime(pr["date"]).dt.normalize()
    tas["date"] = pd.to_datetime(tas["date"]).dt.normalize()
    pet["date"] = pd.to_datetime(pet["date"]).dt.normalize()

    if "pr_bc" not in pr.columns:
        raise ValueError(f"{pr_path} must contain column 'pr_bc'")
    if "tas_bc" not in tas.columns:
        raise ValueError(f"{tas_path} must contain column 'tas_bc'")
    if "pet_mm_day" not in pet.columns:
        raise ValueError(f"{pet_path} must contain column 'pet_mm_day'")

    out = (
        pr[["date", "pr_bc"]]
        .merge(tas[["date", "tas_bc"]], on="date", how="inner")
        .merge(pet[["date", "pet_mm_day"]], on="date", how="inner")
    )
    out = out.rename(columns={"pr_bc": "p_mean", "tas_bc": "t_mean", "pet_mm_day": "pet"})
    return out.sort_values("date").reset_index(drop=True)


def _rainplusmelt_for_basin(df: pd.DataFrame, frac_snow: float) -> np.ndarray:
    met = df[["p_mean", "t_mean"]].copy()
    comp = compute_rainplusmelt_components_for_basin(met, frac_snow=frac_snow)
    return np.asarray(comp["rainplusmelt"], dtype=float)


def build_future_forcing(
    bc_dir: Path,
    forcing_out: Path,
    signatures_csv: Path,
    basins: list[str],
) -> None:
    """Write rainplusmelt.csv and PET.csv (wide: date + one column per basin)."""
    frac_map = get_frac_snow_per_basin(str(signatures_csv), basins)

    rpm_wide: pd.DataFrame | None = None
    pet_wide: pd.DataFrame | None = None
    for b in basins:
        d = _read_bc_forcing(bc_dir, b)
        d["rainplusmelt"] = _rainplusmelt_for_basin(d, frac_map[b])
        rpm_col = d[["date", "rainplusmelt"]].rename(columns={"rainplusmelt": b})
        pet_col = d[["date", "pet"]].rename(columns={"pet": b})
        if rpm_wide is None:
            rpm_wide = rpm_col
            pet_wide = pet_col
        else:
            rpm_wide = rpm_wide.merge(rpm_col, on="date", how="inner")
            pet_wide = pet_wide.merge(pet_col, on="date", how="inner")

    if rpm_wide is None or pet_wide is None:
        raise ValueError("No basins processed.")

    forcing_out.mkdir(parents=True, exist_ok=True)
    rpm_wide.to_csv(forcing_out / "rainplusmelt.csv", index=False)
    pet_wide.to_csv(forcing_out / "PET.csv", index=False)
    print(f"Wrote {forcing_out / 'rainplusmelt.csv'} and {forcing_out / 'PET.csv'}")
    print(f"  Common period: {rpm_wide['date'].min()} .. {rpm_wide['date'].max()} ({len(rpm_wide)} days)")


def run_future_simulation(
    root: Path,
    forcing_out: Path,
    discharge_out: Path,
    basins: list[str],
    warm_span_days: int,
) -> None:
    rain_df = pd.read_csv(forcing_out / "rainplusmelt.csv")
    pet_df = pd.read_csv(forcing_out / "PET.csv")
    rain_df["date"] = pd.to_datetime(rain_df["date"]).dt.normalize()
    pet_df["date"] = pd.to_datetime(pet_df["date"]).dt.normalize()

    cal_df, cal_src = load_preferred_calibration_df(root)
    print(f"Using calibration: {cal_src}")

    forcing_dates = rain_df["date"]
    discharge_out.mkdir(parents=True, exist_ok=True)

    sim_map: dict[str, np.ndarray] = {}
    precip_map: dict[str, np.ndarray] = {}
    pet_map: dict[str, np.ndarray] = {}

    for i_b, basin_id in enumerate(basins):
        if basin_id not in rain_df.columns or basin_id not in pet_df.columns:
            print(f"Skipping {basin_id}: not in forcing columns.")
            continue
        row = cal_df.loc[cal_df["basin_id"] == basin_id]
        if row.empty:
            print(f"Skipping {basin_id}: not in calibration CSV.")
            continue
        row = row.iloc[0]
        precip = rain_df[basin_id].to_numpy(dtype=float)
        pet = pet_df[basin_id].to_numpy(dtype=float)
        params = [
            float(row["alpha1"]),
            float(row["Sm"]),
            float(row["alpha2"]),
            float(row["Kf"]),
            float(row["Ks"]),
        ]
        print(f"Simulating future {i_b + 1}/{len(basins)} ({basin_id}) ...")
        model = DPWM(params, warm_span_days=warm_span_days)
        flux_output, flux_internal, _ = model.simulate(precip, pet)
        sim_map[basin_id] = flux_output.Q
        precip_map[basin_id] = precip
        pet_map[basin_id] = pet

        comp_df = pd.DataFrame(
            {
                "date": forcing_dates,
                "precip": precip,
                "pet": pet,
                "Q_total": flux_output.Q,
                "Q_fast": flux_internal.QF,
                "Q_slow": flux_internal.QS,
                "RF": flux_internal.RF,
                "RS": flux_internal.RS,
                "ET": flux_output.ET,
            }
        )
        comp_out = discharge_out / f"discharge_components_{basin_id}.csv"
        comp_df.to_csv(comp_out, index=False)
        print(f"  Saved {comp_out}")

    q_wide = pd.DataFrame({"date": forcing_dates})
    for basin_id in basins:
        if basin_id in sim_map:
            q_wide[basin_id] = sim_map[basin_id]
    q_path = discharge_out / "Q_total_all_basins.csv"
    q_wide.to_csv(q_path, index=False)
    print(f"Saved {q_path}")


def main() -> None:
    args = parse_args()
    root = args.root.resolve()
    if args.bc_dir is None:
        args.bc_dir = root / "Results" / "eurocordex_bias_corrected"
    if args.forcing_out is None:
        args.forcing_out = root / "Results" / "forcing_future"
    if args.discharge_out is None:
        args.discharge_out = root / "Results" / "discharge_components_future"
    if args.signatures_csv is None:
        args.signatures_csv = root / "data" / "csv_estream" / "estreams_hydrometeo_signatures.csv"

    requested = [b.strip() for b in args.basins.split(",") if b.strip()]

    if args.only_forcing and args.only_simulate:
        raise ValueError("Use at most one of --only_forcing / --only_simulate")

    basins = resolve_basins(
        requested=requested,
        bc_dir=args.bc_dir.resolve(),
        forcing_out=args.forcing_out.resolve(),
        only_simulate=args.only_simulate and not args.only_forcing,
    )

    if not args.only_simulate:
        build_future_forcing(
            bc_dir=args.bc_dir.resolve(),
            forcing_out=args.forcing_out.resolve(),
            signatures_csv=args.signatures_csv.resolve(),
            basins=basins,
        )

    if not args.only_forcing:
        run_future_simulation(
            root=root,
            forcing_out=args.forcing_out.resolve(),
            discharge_out=args.discharge_out.resolve(),
            basins=basins,
            warm_span_days=args.warm_span_days,
        )

    print("Done.")


# Execute the script entry point
main()
